# 01 — lab value (experiment)

This notebook is **not** the live website. The site already shows a lab index, not a transfer fee.

Plan: look for **listed** transfer fees on football-data.org. Train linear regression **only on rows that have a fee**.

`marketValue` is a valuation. Do not treat it as listed Y. If the fee column is empty, **stop**. Do not scrape Transfermarkt. Do not invent Y.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv(Path(".env"))
TOKEN = os.getenv("FOOTBALL_DATA_TOKEN", "").strip()
print("Token loaded:", bool(TOKEN))

Uses `fd_client.py`: 7s spacing, disk cache, quota headers. Free tier is 10 calls/min. Reruns should hit `lab-models/cache/` and spend no extra calls.

In [ ]:
from inspect_listed_fees import main as inspect_fees

inspect_fees()

In [ ]:
import pandas as pd

players = pd.DataFrame(
    columns=["player", "position", "goals", "assists", "minutes", "age", "listed_fee"]
)
players

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

with_fee = players.dropna(subset=["listed_fee"]) if not players.empty else players
if with_fee.empty:
    print("No listed fees. Stop here. Keep the website lab index. This is not a scout's valuation.")
else:
    features = ["goals", "assists", "minutes"]
    model = LinearRegression()
    model.fit(with_fee[features], with_fee["listed_fee"])
    pred = model.predict(with_fee[features])
    mae = mean_absolute_error(with_fee["listed_fee"], pred)
    print(f"MAE on rows that have a listed fee: {mae:.0f}. This is not a scout's valuation.")
    plt.scatter(with_fee["listed_fee"], pred)
    plt.xlabel("Listed fee")
    plt.ylabel("Predicted")
    plt.title("Lab model vs listed fee (not a valuation)")
    plt.show()

**This is not a scout’s valuation.**

On Scout, show “model £X vs listed £Y” **only when Y exists**. It does not exist on this free football-data feed, so the site stays on the lab index.